In [2]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision import transforms
from PIL import Image
import copy
import matplotlib.pyplot as plt
import math

In [ ]:
CROP_SIZE_LR = 48
CROP_SIZE_HR = CROP_SIZE_LR * 4
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "Models"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
train_hr = "Dataset\\Train_hr"
train_x4 = "Dataset\\Train_lr"
valid_hr = "Dataset\\Valid_hr"
valid_x4 = "Dataset\\Valid_lr"

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, num_features):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)
        out = out * 0.1
        return out + residual

class EDSR(nn.Module):
    def __init__(self, num_blocks=32, num_features=256, scale_factor=4):
        super(EDSR, self).__init__()
        self.input_conv = nn.Conv2d(3, num_features, kernel_size=3, padding=1)
        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(num_features) for _ in range(num_blocks)]
        )
        self.output_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.upsample = nn.Conv2d(num_features, 3 * (scale_factor ** 2), kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(scale_factor)

    def forward(self, x):
        x = self.input_conv(x)
        residual = x
        x = self.residual_blocks(x)
        x = self.output_conv(x)
        x += residual
        x = self.upsample(x)
        x = self.pixel_shuffle(x)
        return x

class SuperResolutionDataset(Dataset):
    def __init__(self, hr_dir, lr_dir, crop_size_lr, transform=None, mode='train', scale_factor=4):
        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.crop_size_lr = crop_size_lr
        self.crop_size_hr = crop_size_lr * scale_factor
        self.hr_files = sorted([f for f in os.listdir(hr_dir) if f.endswith('.png')])
        self.lr_files = sorted([f for f in os.listdir(lr_dir) if f.endswith('.png')])
        self.transform = transform
        self.mode = mode
        self.scale_factor = scale_factor

    def __len__(self):
        return len(self.hr_files)

    def __getitem__(self, idx):
        hr_img = Image.open(os.path.join(self.hr_dir, self.hr_files[idx])).convert('RGB')
        lr_img = Image.open(os.path.join(self.lr_dir, self.lr_files[idx])).convert('RGB')
        
        hr_w, hr_h = hr_img.size
        lr_w, lr_h = hr_w // self.scale_factor, hr_h // self.scale_factor

        if self.mode == 'train':
            x = random.randint(0, lr_w - self.crop_size_lr)
            y = random.randint(0, lr_h - self.crop_size_lr)
            hr_crop = hr_img.crop((x * self.scale_factor, y * self.scale_factor, 
                                   (x + self.crop_size_lr) * self.scale_factor, 
                                   (y + self.crop_size_lr) * self.scale_factor))
            lr_crop = lr_img.crop((x, y, x + self.crop_size_lr, y + self.crop_size_lr))
            
            if random.random() > 0.5:
                hr_crop = F.hflip(hr_crop)
                lr_crop = F.hflip(lr_crop)
            if random.random() > 0.5:
                hr_crop = hr_crop.rotate(90)
                lr_crop = lr_crop.rotate(90)
        else:
            x = (lr_w - self.crop_size_lr) // 2
            y = (lr_h - self.crop_size_lr) // 2
            hr_crop = hr_img.crop((x * self.scale_factor, y * self.scale_factor, 
                                   (x + self.crop_size_lr) * self.scale_factor, 
                                   (y + self.crop_size_lr) * self.scale_factor))
            lr_crop = lr_img.crop((x, y, x + self.crop_size_lr, y + self.crop_size_lr))
        
        if self.transform:
            hr_crop = self.transform(hr_crop)
            lr_crop = self.transform(lr_crop)
        
        return lr_crop, hr_crop

def calculate_psnr(output, target, max_pixel_value=1.0):
    mse = torch.mean((output - target) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * math.log10(max_pixel_value / math.sqrt(mse))
    return psnr

def train_and_validate(model, train_loader, val_loader, num_epochs=100):
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999), eps=1e-8)

    best_model_wts = copy.deepcopy(model.state_dict())
    lowest_val_loss = float('inf')

    train_losses = []
    val_losses = []
    val_psnrs = []
    updates = []

    num_updates = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for lr, hr in train_loader:
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            output = model(lr)
            loss = criterion(output, hr)
            train_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            num_updates += 1

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        total_psnr = 0.0
        with torch.no_grad():
            for lr, hr in val_loader:
                lr, hr = lr.to(DEVICE), hr.to(DEVICE)
                output = model(lr)
                loss = criterion(output, hr)
                val_loss += loss.item()
                output_clamped = torch.clamp(output, 0, 1)
                psnr = calculate_psnr(output_clamped, hr)
                total_psnr += psnr

        val_loss /= len(val_loader)
        avg_psnr = total_psnr / len(val_loader)
        val_losses.append(val_loss)
        val_psnrs.append(avg_psnr)
        updates.append(num_updates)

        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Avg PSNR: {avg_psnr:.2f} dB")
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'edsr_x4.pth'))

        if val_loss < lowest_val_loss:
            lowest_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'edsr_x4_best.pth'))
            print(f"Best model updated at epoch {epoch+1}.")

    model.load_state_dict(best_model_wts)
    print("Training complete. Best Validation Loss:", lowest_val_loss)

    plt.figure(figsize=(12, 6))
    plt.plot(updates, train_losses, label="Training Loss", color="blue")
    plt.plot(updates, val_losses, label="Validation Loss", color="orange")
    plt.xlabel("Number of Updates")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss Curve")
    plt.legend()
    plt.grid(True)
    loss_plot_path = os.path.join(CHECKPOINT_DIR, 'loss_curve.png')
    plt.savefig(loss_plot_path)
    plt.close()
    print(f"Loss curve saved to {loss_plot_path}")

    plt.figure(figsize=(12, 6))
    plt.plot(updates, val_psnrs, label="Validation PSNR", color="green")
    plt.xlabel("Number of Updates")
    plt.ylabel("PSNR (dB)")
    plt.title("Validation PSNR Curve")
    plt.legend()
    plt.grid(True)
    psnr_plot_path = os.path.join(CHECKPOINT_DIR, 'psnr_curve.png')
    plt.savefig(psnr_plot_path)
    plt.close()
    print(f"PSNR curve saved to {psnr_plot_path}")

    return model



In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

model_4x = EDSR(scale_factor=4).to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for 4x model.")
    model_4x = nn.DataParallel(model_4x)

#if using transfer learning, load the pretrained model
"""
model = "/kaggle/input/tryingx4/pytorch/default/1/edsr_x4_best.pth"
model_4x.load_state_dict(torch.load(model))
print("Model loaded")
"""

train_dataset_4x = SuperResolutionDataset(train_hr, train_x4, CROP_SIZE_LR, transform, mode='train', scale_factor=4)
val_dataset_4x = SuperResolutionDataset(valid_hr, valid_x4, CROP_SIZE_LR, transform, mode='val', scale_factor=4)

train_loader_4x = DataLoader(train_dataset_4x, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader_4x = DataLoader(val_dataset_4x, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

num_epochs_4x = 900
trained_model_4x = train_and_validate(model_4x, train_loader_4x, val_loader_4x, num_epochs=num_epochs_4x)


Using 2 GPUs for 4x model.


/tmp/ipykernel_23/889496670.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_4x.load_state_dict(torch.load(model_4x_path))


4X Model loaded


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):


Epoch [1/900], Train Loss: 0.0251, Validation Loss: 0.0320, Avg PSNR: 24.95 dB
Best model updated at epoch 1.
Epoch [2/900], Train Loss: 0.0250, Validation Loss: 0.0319, Avg PSNR: 24.96 dB
Best model updated at epoch 2.
Epoch [3/900], Train Loss: 0.0246, Validation Loss: 0.0320, Avg PSNR: 24.96 dB
Epoch [4/900], Train Loss: 0.0244, Validation Loss: 0.0321, Avg PSNR: 24.95 dB
Epoch [5/900], Train Loss: 0.0237, Validation Loss: 0.0320, Avg PSNR: 24.96 dB
Epoch [6/900], Train Loss: 0.0242, Validation Loss: 0.0321, Avg PSNR: 24.94 dB
Epoch [7/900], Train Loss: 0.0249, Validation Loss: 0.0319, Avg PSNR: 24.98 dB
Best model updated at epoch 7.
Epoch [8/900], Train Loss: 0.0250, Validation Loss: 0.0322, Avg PSNR: 24.91 dB
Epoch [9/900], Train Loss: 0.0242, Validation Loss: 0.0320, Avg PSNR: 24.95 dB
Epoch [10/900], Train Loss: 0.0245, Validation Loss: 0.0319, Avg PSNR: 24.98 dB
Best model updated at epoch 10.
Epoch [11/900], Train Loss: 0.0240, Validation Loss: 0.0320, Avg PSNR: 24.94 dB
Epoc